# Prática 10/11 · Experimento honesto: o modelo vence as duas réguas?
**Aprendizagem Computacional · 4º TIA · Aula 03 · versão INICIAL**

Na prática de 03/11, o ETL gerou a `base_analitica.csv` do PPI-molde: 141 municípios de MT, com a abstenção de 2026 (y) e 7 variáveis conhecidas **antes** da eleição (X). Hoje fazemos o **primeiro experimento honesto** com ela, exatamente o que o grupo mostra na sprint review de 17/11:

1. separar o teste **antes de tudo** e guardá-lo;
2. medir duas réguas (baselines): prever a média e repetir 2022;
3. ajustar uma regressão linear dentro de um `Pipeline`;
4. comparar tudo numa tabela, com uma métrica justificada.

**Pergunta:** a regressão linear erra menos que as duas réguas? E as 3 variáveis novas da base (abstenção de 2022, crescimento do eleitorado e distância até Cuiabá) fazem diferença?

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | Separar o teste (20%) antes de tudo | cap. 9 |
| 2 | Separar ajuste e validação dentro do treino | cap. 9 |
| 3 | Baseline da média com `DummyRegressor` | cap. 9 e 11 |
| 4 | Completar o `fit` do baseline de persistência | cap. 9 |
| 5 | Regressão linear dentro de um `Pipeline` | cap. 5 e 11 |

## 1 · Carregar a base analítica (pronto)
A mesma base do PPI-molde. Se a célula falhar no Colab, arraste o CSV da pasta `dados/` para a aba **Arquivos** e rode de novo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
pd.set_option('display.precision', 2)

ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-10_Aula03_Experimento_Honesto/dados/' + ARQUIVO
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL, '|', base.shape)

## 2 · X e y (pronto)
`QT_APTOS` e `QT_ABSTENCOES` são contagens de 2026: com elas dá para **calcular** y. Por isso ficam fora de X (seria vazamento). Também ficam fora os IDs (código e nome).

In [ ]:
Y_COL = 'ABSTENCAO_PCT'
X_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
          'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
X4 = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']   # as 4 do molde antigo
X = base[X_COLS]
y = base[Y_COL]
print('y: média', round(y.mean(), 2), '| desvio', round(y.std(), 2))

## 3 · TODO 1: o teste sai primeiro (cap. 9)
Antes de olhar qualquer número de modelo, separe **20%** dos municípios para teste com `train_test_split(X, y, test_size=0.20, random_state=SEMENTE)`.
O teste fica **guardado** até a prática de 17/11, quando o modelo já estiver escolhido. Hoje ele não é usado.

Pense no inventário florestal: para saber se a equação de volume funciona, você a confere em parcelas que **não** usou para ajustá-la.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO 1: separe 20% para teste, com random_state=SEMENTE
X_treino, X_teste, y_treino, y_teste = None, None, None, None
print('Treino:', len(X_treino), '| Teste (guardado):', len(X_teste))

## 4 · TODO 2: ajuste e validação, dentro do treino (cap. 9)
Se o teste está guardado, onde comparamos os modelos? Numa **validação** tirada do próprio treino. Separe 25% do **treino** (não da base toda) com `random_state=SEMENTE`:
- **ajuste:** onde cada modelo aprende;
- **validação:** onde medimos o erro para comparar.

Em 17/11 vamos trocar essa validação única pela validação cruzada.

In [ ]:
# TODO 2: separe 25% do TREINO para validação, com random_state=SEMENTE
X_ajuste, X_valid, y_ajuste, y_valid = None, None, None, None
print('Ajuste:', len(X_ajuste), '| Validação:', len(X_valid))

## 5 · A métrica (pronto)
Usamos o **MAE** (erro absoluto médio), em pontos percentuais: "o modelo erra, em média, X p.p.". O gestor entende essa frase, e o MAE não é dominado por poucos municípios extremos, como o RMSE. Também mostramos o R² (quanto da variação de y o modelo explica; 0 é o nível de prever a média).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

resultados = {}
def avaliar(nome, previsto):
    resultados[nome] = {'MAE': mean_absolute_error(y_valid, previsto),
                        'RMSE': mean_squared_error(y_valid, previsto) ** 0.5,
                        'R2': r2_score(y_valid, previsto)}
    print(f'{nome:32s} MAE na validação = {resultados[nome]["MAE"]:.2f} p.p.')

## 6 · TODO 3: régua 1, a média (cap. 9 e 11)
Crie `baseline = DummyRegressor(strategy='mean')`, ajuste em `X_ajuste, y_ajuste` e avalie as previsões em `X_valid`.
É a régua de quem não sabe nada sobre o município.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = None  # TODO 3: DummyRegressor(strategy='mean') ajustado em X_ajuste, y_ajuste
avaliar('Baseline: média', baseline.predict(X_valid))

## 7 · TODO 4: régua 2, "a abstenção se repete" (cap. 9)
Um gestor sem modelo diria: "vai faltar quem faltou em 2022". Essa régua é mais difícil de bater que a média.
Como a abstenção do estado pode subir ou cair de uma eleição para a outra, corrigimos 2022 pela variação **média** vista no ajuste:

previsão = abstenção de 2022 + média de (y − abstenção de 2022) no ajuste

A classe abaixo segue o padrão do scikit-learn (`fit` aprende, `predict` prevê), para entrar no mesmo protocolo dos outros modelos. Complete o `fit`: calcule `self.desloc_` como a média de `y - X[self.coluna]`.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class Persistencia(BaseEstimator, RegressorMixin):
    """Baseline "a abstenção se repete": abstenção de 2022 + a variação média vista no treino."""
    def __init__(self, coluna='ABST_2022_PCT'):
        self.coluna = coluna
    def fit(self, X, y):
        self.desloc_ = None   # TODO 4: média de (y - X[self.coluna]); use np.mean e .to_numpy()
        return self
    def predict(self, X):
        return X[self.coluna].to_numpy() + self.desloc_

persistencia = Persistencia().fit(X_ajuste, y_ajuste)
print(f'De 2022 para 2026, a abstenção variou em média {persistencia.desloc_:+.2f} p.p. no ajuste')
avaliar('Baseline: persistência', persistencia.predict(X_valid))

## 8 · TODO 5: a regressão linear, dentro de um `Pipeline` (cap. 5 e 11)
As 7 variáveis estão em escalas muito diferentes (km, %, log). A `StandardScaler` coloca todas em desvios-padrão, o que deixa os coeficientes comparáveis.
Ela vai **dentro** do `Pipeline`: assim a média e o desvio usados na escala são calculados **só no ajuste**, nunca na validação ou no teste.

Crie `linear = Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())])`, ajuste em `X_ajuste, y_ajuste` e avalie em `X_valid`.
A célula também roda, pronta, a mesma regressão só com as 4 variáveis do molde antigo, para comparar.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

linear = None  # TODO 5: Pipeline com StandardScaler e LinearRegression, ajustado em X_ajuste, y_ajuste
avaliar('Regressão linear (7 variáveis)', linear.predict(X_valid))

linear4 = Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]).fit(X_ajuste[X4], y_ajuste)
avaliar('Regressão linear (4 do molde antigo)', linear4.predict(X_valid[X4]))

## 9 · A tabela baseline × modelo (pronto)
É esta tabela que o grupo mostra na sprint review de 17/11.

In [ ]:
tabela = pd.DataFrame(resultados).T.sort_values('MAE')
tabela['ganho sobre a média'] = (1 - tabela['MAE'] / tabela.loc['Baseline: média', 'MAE']).map('{:.0%}'.format)
tabela

**Pergunta para discussão:** com as 4 variáveis do molde antigo, a regressão linear só empata com a persistência, uma regra que nem precisa de modelo. Com as 7, ela ganha das duas réguas com folga. O que isso diz sobre onde estava o problema do molde antigo: no algoritmo ou nos dados?

**Resposta esperada:** nos dados. O algoritmo é o mesmo (regressão linear) nas duas linhas; o que mudou foram as variáveis. Com X fraco, nenhum algoritmo salva o modelo. É por isso que o ETL e a escolha de X (prática de 03/11) pesam tanto quanto a modelagem.

## 10 · Onde o modelo erra? (pronto)
Cada ponto é um município da validação. Na diagonal, acerto exato.

In [ ]:
previsto = linear.predict(X_valid)
fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
eixos[0].scatter(y_valid, previsto, alpha=0.7)
lim = [y_valid.min() - 1, y_valid.max() + 1]
eixos[0].plot(lim, lim, '--', color='grey'); eixos[0].set_xlabel('Abstenção real (%)'); eixos[0].set_ylabel('Prevista (%)')
eixos[0].set_title('Real × previsto (validação)')
coef = pd.Series(linear.named_steps['modelo'].coef_, index=X_COLS).sort_values()
coef.plot.barh(ax=eixos[1], color=['#bf3989' if v < 0 else '#1f6feb' for v in coef])
eixos[1].set_title('Efeito de +1 desvio-padrão (p.p.)'); eixos[1].axvline(0, color='grey', lw=0.8)
plt.tight_layout(); plt.show()
erros = pd.DataFrame({'municipio': base.loc[X_valid.index, 'NM_MUNICIPIO'], 'real': y_valid, 'previsto': previsto})
erros.assign(erro=erros['real'] - erros['previsto']).sort_values('erro', key=abs, ascending=False).head(5)

**Pergunta para discussão:** sozinha, a distância até Cuiabá tem correlação de 0,55 com a abstenção, mas o coeficiente dela no modelo é perto de zero. A distância não importa?

**Resposta esperada:** importa, mas a abstenção de 2022 já carrega essa informação: municípios longe já faltavam mais em 2022. Com 2022 no modelo, a distância não acrescenta nada **novo**. Coeficiente perto de zero num modelo com variáveis correlacionadas não quer dizer "sem efeito".

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('teste separado: 112 no treino e 29 no teste', lambda: len(X_treino) == 112 and len(X_teste) == 29),
    conferir('validação tirada do treino: 84 no ajuste e 28 na validação', lambda: len(X_ajuste) == 84 and len(X_valid) == 28 and set(X_valid.index) <= set(X_treino.index)),
    conferir('baseline da média: MAE 2,42 na validação', lambda: round(resultados['Baseline: média']['MAE'], 2) == 2.42),
    conferir('persistência: deslocamento -2,31 p.p.', lambda: round(persistencia.desloc_, 2) == -2.31),
    conferir('persistência: MAE 2,13 na validação', lambda: round(resultados['Baseline: persistência']['MAE'], 2) == 2.13),
    conferir('linear dentro de Pipeline com escala', lambda: isinstance(linear, Pipeline) and 'escala' in linear.named_steps),
    conferir('linear (7 variáveis): MAE 1,52 na validação', lambda: round(resultados['Regressão linear (7 variáveis)']['MAE'], 2) == 1.52),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as perguntas da banca, para este experimento)
1. **y?** abstenção de 2026 (%), por município. 2. **Baseline?** dois: média e persistência (repetir 2022, corrigido).
3. **Métrica?** MAE em p.p., interpretável pelo gestor. 4. **Split?** teste separado antes de tudo e ainda guardado; comparação numa validação tirada do treino.
5. **Vazamento?** contagens de 2026 fora de X; escala dentro do `Pipeline`. 6. **Evidência?** a tabela da parte 9.
**O que falta (17/11):** uma validação só depende do sorteio. Na próxima prática, trocamos por validação cruzada, comparamos mais modelos e abrimos o teste.